#####  Advanced Build : 

NOTE: Completing this challenge will provide full marks on the assignment, regardless of the completion of the notebook. You do not need to complete this in the notebook for full marks.

MINIMUM REQUIREMENTS:
Baseline LangGraph RAG Application using NAIVE RETRIEVAL
Baseline Evaluation using RAGAS METRICS
Faithfulness
Answer Relevancy
Context Precision
Context Recall
Answer Correctness
Implement a SEMANTIC CHUNKING STRATEGY.
Create an LangGraph RAG Application using SEMANTIC CHUNKING with NAIVE RETRIEVAL.
Compare and contrast results.
SEMANTIC CHUNKING REQUIREMENTS:
Chunk semantically similar (based on designed threshold) sentences, and then paragraphs, greedily, up to a maximum chunk size. Minimum chunk size is a single sentence.

##### Steps involved:
- Baseline LangGraph RAG (naive retrieval + naive chunking)
- Baseline RAGAS eval: Faithfulness, Answer Relevancy, Context Precision, Context Recall, Answer Correctness
- Semantic chunking strategy (greedy: sentences → paragraph-level merge, thresholded, max chunk size, min chunk = 1 sentence)
- LangGraph RAG with semantic chunks (still naive retrieval)
- Compare results (table + short write-up)


##### Step 0 : Setup

In [1]:
import os
from getpass import getpass
from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Please enter your OpenAI API key!")


##### Step 1 : Load data

In [2]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("data/HealthWellnessGuide.txt")
docs = loader.load()


##### Step 2 : Baseline chunking (naive fixed-size)

In [3]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

baseline_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=30)
baseline_chunks = baseline_splitter.split_documents(docs)
len(baseline_chunks)


44

##### Step 3 : Build baseline vector store + naive retriever (k=3)

In [4]:
from langchain_openai import OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

client = QdrantClient(":memory:")
client.create_collection(
    collection_name="baseline_chunks",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)

baseline_vs = QdrantVectorStore(
    client=client,
    collection_name="baseline_chunks",
    embedding=embeddings,
)

_ = baseline_vs.add_documents(baseline_chunks)

baseline_retriever = baseline_vs.as_retriever(search_kwargs={"k": 3})


##### Step 4 : Baseline LangGraph RAG (retrieve → generate)

In [5]:
from langchain.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langgraph.graph import START, StateGraph
from typing_extensions import TypedDict
from typing import List
from langchain_core.documents import Document

RAG_PROMPT = """\
You are a helpful assistant who answers questions based on provided context.
You MUST only use the provided context.

### Question
{question}

### Context
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)
llm = ChatOpenAI(model="gpt-4.1-nano")

class State(TypedDict):
    question: str
    context: List[Document]
    response: str

def retrieve_baseline(state: State):
    retrieved_docs = baseline_retriever.invoke(state["question"])
    return {"context": retrieved_docs}

def generate(state: State):
    docs_content = "\n\n".join(doc.page_content for doc in state["context"])
    messages = rag_prompt.format_messages(question=state["question"], context=docs_content)
    response = llm.invoke(messages)
    return {"response": response.content}

baseline_graph = StateGraph(State).add_sequence([retrieve_baseline, generate])
baseline_graph.add_edge(START, "retrieve_baseline")
baseline_app = baseline_graph.compile()


##### Step 5 : Create eval dataset (use your existing synthetic dataset if you already generated it)

In [6]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.testset import TestsetGenerator

from langchain_openai import ChatOpenAI

generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))

gen = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = gen.generate_with_langchain_docs(docs, testset_size=20)


# Run dataset 
import copy
from ragas import EvaluationDataset

baseline_ds = copy.deepcopy(dataset)

for row in baseline_ds:
    out = baseline_app.invoke({"question": row.eval_sample.user_input})
    row.eval_sample.response = out["response"]
    row.eval_sample.retrieved_contexts = [c.page_content for c in out["context"]]

baseline_eval_dataset = EvaluationDataset.from_pandas(baseline_ds.to_pandas())


Applying HeadlinesExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/1 [00:00<?, ?it/s]

Applying SummaryExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/4 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/9 [00:00<?, ?it/s]

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/1 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/2 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/19 [00:00<?, ?it/s]

##### Step 6 : Baseline RAGAS evaluation (required metrics)

In [7]:
from ragas import evaluate, RunConfig
from ragas.llms import LangchainLLMWrapper

# Metrics (handle version differences safely)
from ragas.metrics import Faithfulness, ResponseRelevancy

try:
    from ragas.metrics import ContextPrecision, ContextRecall, AnswerCorrectness
except Exception:
    # Fallback names in some versions
    from ragas.metrics import LLMContextRecall as ContextRecall
    # If ContextPrecision/AnswerCorrectness not available, keep requirement set but you may need to upgrade ragas.
    # We'll still try to import later if available.
    ContextPrecision = None
    AnswerCorrectness = None

evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))

metrics = [Faithfulness(), ResponseRelevancy(), ContextRecall()]

if ContextPrecision is not None:
    metrics.append(ContextPrecision())
if AnswerCorrectness is not None:
    metrics.append(AnswerCorrectness())

run_cfg = RunConfig(timeout=360)

baseline_result = evaluate(
    dataset=baseline_eval_dataset,
    metrics=metrics,
    llm=evaluator_llm,
    run_config=run_cfg
)

baseline_result


Evaluating:   0%|          | 0/95 [00:00<?, ?it/s]

{'faithfulness': 0.6592, 'answer_relevancy': 0.8026, 'context_recall': 0.5768, 'context_precision': 0.5307, 'answer_correctness': 0.6400}

##### SEMANTIC CHUNKING IMPLEMENTATION (Meets their requirements)
What it does
- Split into paragraphs
- Split each paragraph into sentences
- Compute embeddings per sentence
- Greedy sentence chunking: append sentence if similarity ≥ threshold and size ≤ max_chars, else start new chunk
- Greedy paragraph-level merge: merge adjacent chunks if similarity ≥ threshold_par and size ≤ max_chars
- Minimum chunk = one sentence


##### Step 7 : Semantic chunking code

In [8]:
import re
import numpy as np
from dataclasses import dataclass
from typing import List, Tuple
from langchain_core.documents import Document

def split_paragraphs(text: str) -> List[str]:
    paras = [p.strip() for p in re.split(r"\n\s*\n+", text) if p.strip()]
    return paras

def split_sentences(paragraph: str) -> List[str]:
    # Simple sentence split (good enough for this assignment)
    sents = re.split(r"(?<=[.!?])\s+", paragraph.strip())
    sents = [s.strip() for s in sents if s.strip()]
    return sents

def cosine_sim(a: np.ndarray, b: np.ndarray) -> float:
    denom = (np.linalg.norm(a) * np.linalg.norm(b))
    if denom == 0:
        return 0.0
    return float(np.dot(a, b) / denom)

def mean_embed(vecs: List[np.ndarray]) -> np.ndarray:
    return np.mean(np.stack(vecs, axis=0), axis=0)

@dataclass
class ChunkObj:
    text: str
    emb: np.ndarray

def semantic_chunk_text(
    text: str,
    embed_fn,
    sim_threshold_sent: float = 0.78,
    sim_threshold_para: float = 0.75,
    max_chars: int = 1200
) -> List[ChunkObj]:
    """
    Requirements met:
    - greedy sentence chunking by semantic similarity threshold up to max chunk size
    - then greedy merging across paragraphs/chunks (paragraph-level)
    - min chunk size = single sentence
    """
    chunks: List[ChunkObj] = []

    paragraphs = split_paragraphs(text)
    for para in paragraphs:
        sents = split_sentences(para)
        if not sents:
            continue

        sent_embs = embed_fn(sents)  # list of vectors
        sent_embs = [np.array(v, dtype=np.float32) for v in sent_embs]

        current_sents = [sents[0]]
        current_embs = [sent_embs[0]]

        for sent, emb in zip(sents[1:], sent_embs[1:]):
            curr_text = " ".join(current_sents)
            curr_emb = mean_embed(current_embs)

            sim = cosine_sim(curr_emb, emb)
            if (sim >= sim_threshold_sent) and (len(curr_text) + 1 + len(sent) <= max_chars):
                current_sents.append(sent)
                current_embs.append(emb)
            else:
                # close current chunk
                chunk_text = " ".join(current_sents).strip()
                chunks.append(ChunkObj(text=chunk_text, emb=mean_embed(current_embs)))
                # start new chunk (min = single sentence)
                current_sents = [sent]
                current_embs = [emb]

        # close last chunk in paragraph
        chunk_text = " ".join(current_sents).strip()
        chunks.append(ChunkObj(text=chunk_text, emb=mean_embed(current_embs)))

    # Paragraph-level / cross-chunk greedy merge (adjacent)
    merged: List[ChunkObj] = []
    i = 0
    while i < len(chunks):
        cur = chunks[i]
        j = i + 1
        # greedily merge forward while similar and size allows
        while j < len(chunks):
            nxt = chunks[j]
            sim = cosine_sim(cur.emb, nxt.emb)
            if sim >= sim_threshold_para and (len(cur.text) + 2 + len(nxt.text) <= max_chars):
                new_text = (cur.text + "\n\n" + nxt.text).strip()
                new_emb = mean_embed([cur.emb, nxt.emb])
                cur = ChunkObj(text=new_text, emb=new_emb)
                j += 1
            else:
                break
        merged.append(cur)
        i = j

    return merged

# Helper: embed list of strings using LangChain embeddings
def embed_texts(texts: List[str]) -> List[List[float]]:
    return embeddings.embed_documents(texts)

#Build semantic chunks as LangChain Documents:
semantic_docs: List[Document] = []

for d in docs:
    chs = semantic_chunk_text(
        d.page_content,
        embed_fn=embed_texts,
        sim_threshold_sent=0.78,
        sim_threshold_para=0.75,
        max_chars=1200
    )
    for c in chs:
        semantic_docs.append(Document(page_content=c.text, metadata=d.metadata))

len(semantic_docs)


161

##### Step 8 :Semantic chunk RAG (still naive retrieval)

In [9]:
client2 = QdrantClient(":memory:")
client2.create_collection(
    collection_name="semantic_chunks",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)

semantic_vs = QdrantVectorStore(
    client=client2,
    collection_name="semantic_chunks",
    embedding=embeddings,
)

_ = semantic_vs.add_documents(semantic_docs)

semantic_retriever = semantic_vs.as_retriever(search_kwargs={"k": 3})

def retrieve_semantic(state: State):
    retrieved_docs = semantic_retriever.invoke(state["question"])
    return {"context": retrieved_docs}

semantic_graph = StateGraph(State).add_sequence([retrieve_semantic, generate])
semantic_graph.add_edge(START, "retrieve_semantic")
semantic_app = semantic_graph.compile()

# Run eval dataset through semantic app:
semantic_ds = copy.deepcopy(dataset)

for row in semantic_ds:
    out = semantic_app.invoke({"question": row.eval_sample.user_input})
    row.eval_sample.response = out["response"]
    row.eval_sample.retrieved_contexts = [c.page_content for c in out["context"]]

semantic_eval_dataset = EvaluationDataset.from_pandas(semantic_ds.to_pandas())

#Evaluate:
semantic_result = evaluate(
    dataset=semantic_eval_dataset,
    metrics=metrics,
    llm=evaluator_llm,
    run_config=run_cfg
)

semantic_result


Evaluating:   0%|          | 0/95 [00:00<?, ?it/s]

{'faithfulness': 0.5938, 'answer_relevancy': 0.5605, 'context_recall': 0.3346, 'context_precision': 0.7105, 'answer_correctness': 0.5071}

##### Step 9 : Compare results (required “compare & contrast”)

In [13]:
import pandas as pd
import numpy as np

def result_to_scores(res):
    return res._scores_dict

baseline_dict = result_to_scores(baseline_result)
semantic_dict = result_to_scores(semantic_result)

# Convert list values → scalar means (if needed)
def normalize_scores(score_dict):
    normalized = {}
    for k, v in score_dict.items():
        if isinstance(v, list):
            normalized[k] = float(np.mean(v))
        else:
            normalized[k] = float(v)
    return normalized

baseline_dict = normalize_scores(baseline_dict)
semantic_dict = normalize_scores(semantic_dict)

df = pd.DataFrame([
    {"system": "baseline_naive_chunks", **baseline_dict},
    {"system": "semantic_chunks", **semantic_dict},
])

# Delta row
common_metrics = [c for c in df.columns if c != "system"]
delta = {"system": "delta(semantic-baseline)"}

for m in common_metrics:
    delta[m] = (
        df.loc[df["system"]=="semantic_chunks", m].values[0]
        - df.loc[df["system"]=="baseline_naive_chunks", m].values[0]
    )

df2 = pd.concat([df, pd.DataFrame([delta])], ignore_index=True)

df2


,system,faithfulness,answer_relevancy,context_recall,context_precision,answer_correctness
0,baseline_naive_chunks,0.659240,0.802623,0.576754,0.530702,0.640026
1,semantic_chunks,0.593813,0.560541,0.334649,0.710526,0.507079
2,delta(semantic-baseline),-0.065427,-0.242082,-0.242105,0.179825,-0.132948


#### Findings of the Activity

###### Compare & Contrast: Baseline vs Semantic Chunking

The semantic chunking strategy improved **context precision** significantly (+0.18), indicating that retrieved chunks were more internally coherent and less noisy. This confirms that semantically grouped sentences produce higher-quality individual chunks.

However, semantic chunking reduced **context recall (-0.24)** and consequently decreased answer relevancy and answer correctness. Because the retriever used naive retrieval with k=3, fewer semantically dense chunks were retrieved, limiting total information coverage.

Faithfulness also dropped slightly, likely due to reduced supporting context leading to minor model generalization.

Overall, semantic chunking improved precision but reduced coverage under naive retrieval. This suggests that semantic chunking performs best when paired with higher k values or reranking strategies to recover recall while preserving precision.
